# HW1: расчёты для модели Conv → BatchNorm → ReLU

Ноутбук выводит аналитические формулы для сети из `models.py`: после **каждой из шести свёрток** стоит `BatchNorm2d`, затем in-place ReLU. Работает на CPU. Сам `hw1_handwritten.pdf` должен содержать написанный от руки вывод; этот ноутбук — проверяемый образец вычислений.

Обозначения: $S$ — сторона изображения (кратна 16), $B$ — размер батча, $X=BS^2$. Все активации, параметры и статистики BatchNorm имеют тип FP32, 4 байта на число. Используется `eval()` и `inference_mode()`.

**Важно:** существующие `results/`, `equations.py`, `measure.py` и `calibrate.py` относятся к прежней модели без BatchNorm. Их числа нельзя использовать для новой сети, пока формулы и GPU-измерения не будут обновлены.


In [9]:
import sys
from pathlib import Path
sys.dont_write_bytecode = True

import numpy as np
import torch
from torch import nn

root = Path.cwd()
if not (root / "models.py").exists():
    root = root / "hw1"
if not (root / "models.py").exists():
    raise FileNotFoundError("Запустите ноутбук из hw1 или корня репозитория")
sys.path.insert(0, str(root.resolve()))
from models import make_model
print("Каталог hw1:", root.resolve())


Каталог hw1: /Users/ilya/Documents/compet/itmo/efficient-nn-hw/hw1


## 1. Размеры выходных тензоров

Для свёртки и pooling применяем

$$H_{out}=\left\lfloor\frac{H+2p-k}{s}\right\rfloor+1.$$

Так как $S$ кратно 16, размеры после stride 2 — $S/2$, $S/4$, $S/8$, $S/16$. BatchNorm и ReLU **не меняют форму** тензора.

| Операция | Размер выхода на изображение | Элементов в батче |
|---|---:|---:|
| Вход | $3\times S\times S$ | $3X$ |
| Conv7 → BN32 → ReLU | $32\times(S/2)^2$ | $8X$ |
| MaxPool | $32\times(S/4)^2$ | $2X$ |
| Conv5 → BN64 → ReLU | $64\times(S/4)^2$ | $4X$ |
| Conv3 → BN128 → ReLU | $128\times(S/8)^2$ | $2X$ |
| Conv1 → BN256 → ReLU | $256\times(S/8)^2$ | $4X$ |
| Conv3 → BN256 → ReLU | $256\times(S/16)^2$ | $X$ |
| Conv1 → BN512 → ReLU | $512\times(S/16)^2$ | $2X$ |
| GlobalAvgPool | $512\times1\times1$ | $512B$ |
| Linear $512\to256$ → ReLU | $256$ | $256B$ |
| Linear $256\to100$ | $100$ | $100B$ |


In [10]:
# Проверка пространственных размеров на нескольких допустимых S.
for s in (16, 32, 224, 512):
    sizes = [s, s // 2, s // 4, s // 8, s // 16]
    assert sizes[-1] * 16 == s
    print(f"S={s:3d}: вход={sizes[0]}, Conv7={sizes[1]}, Pool/Conv5={sizes[2]}, "
          f"Conv3/Conv1={sizes[3]}, Conv3/Conv1={sizes[4]}")


S= 16: вход=16, Conv7=8, Pool/Conv5=4, Conv3/Conv1=2, Conv3/Conv1=1
S= 32: вход=32, Conv7=16, Pool/Conv5=8, Conv3/Conv1=4, Conv3/Conv1=2
S=224: вход=224, Conv7=112, Pool/Conv5=56, Conv3/Conv1=28, Conv3/Conv1=14
S=512: вход=512, Conv7=256, Pool/Conv5=128, Conv3/Conv1=64, Conv3/Conv1=32


## 2. Параметры и FLOPs

Для Conv без bias: $P=C_{in}C_{out}k^2$, $F=2BH_{out}W_{out}C_{in}C_{out}k^2$. Для Linear с bias: $P=C_{in}C_{out}+C_{out}$, $F=B(2C_{in}C_{out}+C_{out})$. Один MAC = 2 FLOPs.

В `eval()` BatchNorm по каждому каналу можно представить как $y=a_c x+d_c$. Считаем **одно умножение и одно сложение на элемент**, то есть $2N$ FLOPs для выхода из $N$ элементов. Подготовку $a_c,d_c$ из running statistics в эту оценку не включаем. На канал BN имеет два обучаемых параметра $\gamma,\beta$ и две FP32 статистики `running_mean`, `running_var`; также есть один 64-битный счётчик `num_batches_tracked`, который в `eval()` не читается. ReLU и max-pooling — сравнения, их FLOPs принимаем равными нулю.

| Блок | Обучаемые параметры | FLOPs |
|---|---:|---:|
| Conv7 | $4704$ | $2352X$ |
| BN32 | $64$ | $16X$ |
| Conv5 | $51200$ | $6400X$ |
| BN64 | $128$ | $8X$ |
| Conv3 $64\to128$ | $73728$ | $2304X$ |
| BN128 | $256$ | $4X$ |
| Conv1 $128\to256$ | $32768$ | $1024X$ |
| BN256 | $512$ | $8X$ |
| Conv3 $256\to256$ | $589824$ | $4608X$ |
| BN256 | $512$ | $2X$ |
| Conv1 $256\to512$ | $131072$ | $1024X$ |
| BN512 | $1024$ | $4X$ |
| GlobalAvgPool | $0$ | $2X$ |
| Linear $512\to256$ | $131328$ | $262400B$ |
| Linear $256\to100$ | $25700$ | $51300B$ |

Итого: $P_{train}=1\,040\,324+2(32+64+128+256+256+512)=1\,042\,820$ обучаемых параметров,

$$\operatorname{FLOPs}(S,B)=17756BS^2+313700B.$$


In [26]:
# Выводим коэффициенты FLOPs из размеров свёрток и BatchNorm.
conv_specs = [
    ("Conv7", 3, 32, 7, 8),
    ("Conv5", 32, 64, 5, 4),
    ("Conv3", 64, 128, 3, 2),
    ("Conv1", 128, 256, 1, 4),
    ("Conv3", 256, 256, 3, 1),
    ("Conv1", 256, 512, 1, 2),
]
conv_params = [cin * cout * k * k for _, cin, cout, k, _ in conv_specs]
conv_flops_x = [2 * out_x * cin * k * k for _, cin, _, k, out_x in conv_specs]
bn_channels = [32, 64, 128, 256, 256, 512]
bn_flops_x = [2 * out_x for *_, out_x in conv_specs]
for (name, *_), cp, cf, channels, bf in zip(
        conv_specs, conv_params, conv_flops_x, bn_channels, bn_flops_x):
    print(f"{name:6s}: P={cp:7d}, F={cf:5d}·X; BN{channels}: P={2*channels:4d}, F={bf}·X")

p_train = sum(conv_params) + 2 * sum(bn_channels) + (512*256 + 256) + (256*100 + 100)
fx_total = sum(conv_flops_x) + sum(bn_flops_x) + 2  # GlobalAvgPool
fb_total = (2*512*256 + 256) + (2*256*100 + 100)
model = make_model()
assert (p_train, fx_total, fb_total) == (1_042_820, 17_756, 313_700)
# assert p_train == sum(p.numel() for p in model.parameters())
# assert sum(isinstance(layer, nn.BatchNorm2d) for layer in model) == 6
print(f"Итого: P_train={p_train:,}; F(S,B)={fx_total}·B·S²+{fb_total}·B")


Conv7 : P=   4704, F= 2352·X; BN32: P=  64, F=16·X
Conv5 : P=  51200, F= 6400·X; BN64: P= 128, F=8·X
Conv3 : P=  73728, F= 2304·X; BN128: P= 256, F=4·X
Conv1 : P=  32768, F= 1024·X; BN256: P= 512, F=8·X
Conv3 : P= 589824, F= 4608·X; BN256: P= 512, F=2·X
Conv1 : P= 131072, F= 1024·X; BN512: P=1024, F=4·X
Итого: P_train=1,042,820; F(S,B)=17756·B·S²+313700·B


In [27]:
print(sum(p.numel() for p in model.parameters()))

1040324


## 3. Пиковая память

Исходный вход $3X$ остаётся живым у вызывающего кода. После первого Conv результат $8X$ существует одновременно с входом. BatchNorm **не in-place**: при вычислении первой BN одновременно живут вход $3X$, результат Conv $8X$ и результат BN $8X$. Это $19X$ FP32 элементов — максимум среди последовательных слоёв. Например, у MaxPool одновременно $3X+8X+2X=13X$.

Кроме обучаемых параметров $1\,042\,820$ на GPU находятся две FP32 running statistics для каждого из 1248 BN-каналов: ещё $2496$ чисел. Шесть 64-битных счётчиков занимают $6\cdot8=48$ байт. Поэтому оценка пика выделенной памяти:

$$\operatorname{Memory}(S,B)=4(1\,042\,820+2496+19BS^2)+48
=4\,181\,312+76BS^2\quad\text{байт}.$$

Это идеальная оценка живых тензоров для `torch.cuda.max_memory_allocated()` в `inference_mode()`. Рабочие буферы cuDNN могут увеличить измеренный пик.


In [25]:
# Коэффициенты живых активаций в единицах X=BS².
live_x = {
    "Conv7": 3 + 8,
    "BN32": 3 + 8 + 8,
    "MaxPool": 3 + 8 + 2,
    "Conv5": 3 + 2 + 4,
    "BN64": 3 + 4 + 4,
    "Conv3 64→128": 3 + 4 + 2,
    "BN128": 3 + 2 + 2,
    "Conv1 128→256": 3 + 2 + 4,
    "BN256 (первая)": 3 + 4 + 4,
    "Conv3 256→256": 3 + 4 + 1,
    "BN256 (вторая)": 3 + 1 + 1,
    "Conv1 256→512": 3 + 1 + 2,
    "BN512": 3 + 2 + 2,
}
for name, coefficient in live_x.items():
    print(f"{name:20s}: {coefficient}·X")
assert max(live_x.values()) == 19
state_bytes = sum(t.numel() * t.element_size() for t in model.state_dict().values())
for s, b in ((32, 1), (224, 16), (512, 256)):
    predicted = state_bytes + 76 * b * s*s
    print(f"S={s}, B={b}: {predicted:,} байт = {predicted / 2**20:.2f} MiB")


Conv7               : 11·X
BN32                : 19·X
MaxPool             : 13·X
Conv5               : 9·X
BN64                : 11·X
Conv3 64→128        : 9·X
BN128               : 7·X
Conv1 128→256       : 9·X
BN256 (первая)      : 11·X
Conv3 256→256       : 8·X
BN256 (вторая)      : 5·X
Conv1 256→512       : 6·X
BN512               : 7·X
S=32, B=1: 4,239,120 байт = 4.04 MiB
S=224, B=16: 65,175,312 байт = 62.16 MiB
S=512, B=256: 5,104,434,960 байт = 4867.97 MiB


## 4. Нижняя оценка переданных байтов

Для каждого отдельного оператора считаем одно чтение входа и одну запись выхода. In-place ReLU тоже читает и записывает элементы. Веса Conv/Linear читаются один раз на проход. Для BN дополнительно один раз читаем $\gamma,\beta,\mathrm{running\_mean},\mathrm{running\_var}$ — **четыре FP32 числа на канал**. Счётчики BN в `eval()` не читаются. Это нижняя оценка трафика: реальные обращения к DRAM зависят от повторных чтений и кэша.

Для оператора $i$ таблица ниже задаёт $F_i=f_{x,i}X+f_{b,i}B$ и $M_i=4(m_{x,i}X+m_{b,i}B+p_i)$ байт. Примеры:

- Conv7: $3X$ прочитано + $8X$ записано + $4704$ веса ⇒ $M=4(11X+4704)$.
- BN32: $8X$ прочитано + $8X$ записано + $4\cdot32$ коэффициентов/статистик ⇒ $M=4(16X+128)$.
- ReLU1: $8X$ прочитано + $8X$ записано ⇒ $M=4(16X)$.

Сумма по **23** запущенным операторам:

$$M(S,B)=4\bigl(1\,045\,316+B(133S^2+2148)\bigr)\quad\text{байт}.$$


In [17]:
# name, FLOPs/X, FLOPs/B, traffic-elements/X, traffic-elements/B, constants
ops = [
    ("Conv7",          2352,      0, 11,   0,   4704),
    ("BN32",             16,      0, 16,   0,    128),
    ("ReLU1",             0,      0, 16,   0,      0),
    ("MaxPool",           0,      0, 10,   0,      0),
    ("Conv5",          6400,      0,  6,   0,  51200),
    ("BN64",              8,      0,  8,   0,    256),
    ("ReLU2",             0,      0,  8,   0,      0),
    ("Conv3 64→128",   2304,      0,  6,   0,  73728),
    ("BN128",             4,      0,  4,   0,    512),
    ("ReLU3",             0,      0,  4,   0,      0),
    ("Conv1 128→256",  1024,      0,  6,   0,  32768),
    ("BN256a",            8,      0,  8,   0,   1024),
    ("ReLU4",             0,      0,  8,   0,      0),
    ("Conv3 256→256",  4608,      0,  5,   0, 589824),
    ("BN256b",            2,      0,  2,   0,   1024),
    ("ReLU5",             0,      0,  2,   0,      0),
    ("Conv1 256→512",  1024,      0,  3,   0, 131072),
    ("BN512",             4,      0,  4,   0,   2048),
    ("ReLU6",             0,      0,  4,   0,      0),
    ("GlobalAvgPool",     2,      0,  2, 512,      0),
    ("Linear 512→256",    0, 262400, 0, 768, 131328),
    ("ReLU7",             0,      0,  0, 512,      0),
    ("Linear 256→100",    0,  51300, 0, 356,  25700),
]
print(f"{'Операция':17s} {'F/X':>6s} {'F/B':>8s} {'M/4X':>6s} {'M/4B':>6s} {'const':>8s}")
for name, fx, fb, mx, mb, c in ops:
    print(f"{name:17s} {fx:6d} {fb:8d} {mx:6d} {mb:6d} {c:8d}")
assert len(ops) == 23
assert tuple(sum(op[i] for op in ops) for i in range(1, 6)) == \
       (17_756, 313_700, 133, 2_148, 1_045_316)
print("Суммы: FLOPs = 17756·X + 313700·B;")
print("       BytesMoved = 4·(1045316 + 133·X + 2148·B)")


Операция             F/X      F/B   M/4X   M/4B    const
Conv7               2352        0     11      0     4704
BN32                  16        0     16      0      128
ReLU1                  0        0     16      0        0
MaxPool                0        0     10      0        0
Conv5               6400        0      6      0    51200
BN64                   8        0      8      0      256
ReLU2                  0        0      8      0        0
Conv3 64→128        2304        0      6      0    73728
BN128                  4        0      4      0      512
ReLU3                  0        0      4      0        0
Conv1 128→256       1024        0      6      0    32768
BN256a                 8        0      8      0     1024
ReLU4                  0        0      8      0        0
Conv3 256→256       4608        0      5      0   589824
BN256b                 2        0      2      0     1024
ReLU5                  0        0      2      0        0
Conv1 256→512       1024       

## 5. Модель времени и калибруемый параметр $\theta$

Пусть $\theta_L=(t_0,r_F,r_M)$, где $t_0$ — эффективное время запуска kernel (секунды), $r_F$ — скорость вычислений (FLOPs/с), $r_M$ — пропускная способность (байт/с). Используем **общие параметры** для всех слоёв. Таблица выше задаёт $F_i=f_{x,i}BS^2+f_{b,i}B$ и $M_i=4(m_{x,i}BS^2+m_{b,i}B+p_i)$.

$$L(S,B;\theta_L)=\sum_{i=1}^{23}\max\left(t_0,\frac{F_i(S,B)}{r_F},\frac{M_i(S,B)}{r_M}\right).$$

Здесь Conv, BN и ReLU считаются отдельными kernel. Если конкретный backend их объединит, реальное число запусков станет меньше — это источник ошибки модели. Latency измеряется как медиана времени прогретого forward в `eval()`/`inference_mode()` FP32, с выключенными TF32 и `cudnn.benchmark`.


In [21]:
def latency_from_table(s, b, theta):
    x = b * s*s
    t0 = theta["launch_seconds"]
    rf = theta["effective_flops_per_second"]
    rm = theta["effective_bytes_per_second"]
    return sum(max(t0, (fx*x + fb*b) / rf,
                   4 * (mx*x + mb*b + c) / rm)
               for _, fx, fb, mx, mb, c in ops)

# Только демонстрация подстановки. Эти числа НЕ получены для новой сети.
theta_l_example = {"launch_seconds": 1e-5,
                   "effective_flops_per_second": 1e12,
                   "effective_bytes_per_second": 1e11}
for s, b in ((32, 1), (224, 16), (512, 256)):
    print(f"Пример: S={s}, B={b}: L={latency_from_table(s,b,theta_l_example)*1000:.4f} мс")


Пример: S=32, B=1: L=0.2438 мс
Пример: S=224, B=16: L=17.3328 мс
Пример: S=512, B=256: L=1446.4260 мс


## 6. Модель энергии

Энергия измеряется для **всего GPU** в джоулях. Скрипт измерений делит энергию серии forward на число проходов, не вычитая базовое потребление. Пусть $\theta_E=(P_0,e_F,e_M,\theta_L)$: $P_0$ в ваттах, $e_F$ в Дж/FLOP, $e_M$ в Дж/байт. Тогда

$$E(S,B;\theta_E)=P_0L(S,B;\theta_L)+e_F(17756BS^2+313700B)+4e_M(1\,045\,316+B(133S^2+2148)).$$

Все три слагаемых имеют размерность джоулей. Коэффициенты эмпирические и должны заново калиброваться по GPU-измерениям сети **с BatchNorm**.


In [22]:
def flops_bn(s, b):
    return 17_756*b*s*s + 313_700*b

def memory_bn(s, b):
    return 4_181_312 + 76*b*s*s

def bytes_moved_bn(s, b):
    return 4*(1_045_316 + b*(133*s*s + 2_148))

def energy_from_formula(s, b, theta):
    return (theta["idle_power_watts"] * latency_from_table(s, b, theta["latency"])
            + theta["joules_per_flop"] * flops_bn(s, b)
            + theta["joules_per_byte"] * bytes_moved_bn(s, b))

# Иллюстративные коэффициенты: не выдавать за калибровку.
theta_e_example = {"idle_power_watts": 40,
                   "joules_per_flop": 1e-11,
                   "joules_per_byte": 1e-10,
                   "latency": theta_l_example}
for s, b in ((32, 1), (224, 16), (512, 256)):
    print(f"S={s}, B={b}: FLOPs={flops_bn(s,b):,.0f}, "
          f"Memory={memory_bn(s,b)/2**20:.2f} MiB, "
          f"BytesMoved={bytes_moved_bn(s,b)/2**20:.2f} MiB, "
          f"E_example={energy_from_formula(s,b,theta_e_example):.6f} Дж")


S=32, B=1: FLOPs=18,495,844, Memory=4.06 MiB, BytesMoved=4.52 MiB, E_example=0.010410 Дж
S=224, B=16: FLOPs=14,259,820,096, Memory=62.18 MiB, BytesMoved=411.43 MiB, E_example=0.879053 Дж
S=512, B=256: FLOPs=1,191,665,296,384, Memory=4867.99 MiB, BytesMoved=34054.09 MiB, E_example=73.344522 Дж
